# Precompute Evaluated Cross Sections (Hybrid Approach)

This notebook precomputes hybrid angular distributions (c0 from EXFOR fit + a_l from ENDF MF4)
for JEFF-4.0, JENDL-5, and This Work, then saves the results to a parquet file for chi-squared analysis.

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────

MT_NUMBER = 2

# ENDF evaluation files
JEFF_FILE  = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/26-Fe-56g.txt"
JENDL_FILE = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/260560.jendl5"
USER_FILE  = "/share_snc/snc/JuanMonleon/ENDF_samples/new_test_50/26-Fe-56g_nominal_mg.endf"
# Optional: capped order 3 without cierjacks
USER2_FILE = None #"/share_snc/snc/JuanMonleon/ENDF_samples/without-cierjacks_cap3/26-Fe-56g_nominal_mg.endf" 
# Optional: capped order 3 with cierjacks
USER3_FILE = None #"/share_snc/snc/JuanMonleon/ENDF_samples/cap3/26-Fe-56g_nominal_mg.endf"

# EXFOR configuration
EXFOR_DB_PATH = '/share_snc/snc/JuanMonleon/EXFOR/x4_iron_angular.db'
TARGET_ZAIDS  = [26056, 26000]  # Fe-56 + natural iron

SUPPLEMENTARY_JSON_FILES = [
    '/share_snc/snc/JuanMonleon/EXFOR/data_v1/27673002.json',
]

EXCLUDE_EXPERIMENTS = ["32246002"]
# - "20743002" - Cierjacks (1978)
# - "32246002" - Tostkii (1957)

# Normalization rules: which subentry to use for c0 fit per library
KINNEY_SUBENTRY      = "10571002"   # used when E < ENERGY_THRESHOLD_MEV
SMITH_SUBENTRY       = "10886002"   # used when E >= ENERGY_THRESHOLD_MEV
ENERGY_THRESHOLD_MEV = 2.5

# Physics parameters
M_PROJ_U = 1.008665   # neutron mass (u)
M_TARG_U = 55.93494   # Fe-56 mass (u)

MIN_STAT_RELATIVE_UNCERTAINTY = 0.01  # statistical-only relative floor (matches pipeline)

# GENDF cross-section covariance files (MF33, multigroup) — used to add the
# integral cross-section uncertainty to the evaluation σ at each angular point.
# Set to None for libraries without an available GENDF.
JEFF_XS_GENDF  = '/soft_snc/lib/cov/40/293.6/260560_40.02.xs.gendf'
JENDL_XS_GENDF = '/share_snc/snc/JuanMonleon/COV/cov/50/600/260560_50.06.xs.gendf'
USER_XS_GENDF  = '/soft_snc/lib/cov/40/293.6/260560_40.02.xs.gendf' 
USER2_XS_GENDF = None
USER3_XS_GENDF = None

# GENDF cross-section covariance files (multigroup, MT=2 self-block).
# Used to add the integral cross-section uncertainty to σ_eval at each
# angular point. Multigroup is an acceptable approximation here: it avoids
# the MF33 NC LTY=0 reconstruction (sum-rule resolution for redundant MTs)
# that pointwise ENDF parsing requires. Set to None for libraries without
# an available GENDF.
JEFF_XS_GENDF  = '/soft_snc/lib/cov/40/293.6/260560_40.02.xs.gendf'
JENDL_XS_GENDF = '/share_snc/snc/JuanMonleon/COV/cov/50/600/260560_50.06.xs.gendf'
USER_XS_GENDF  = None
USER2_XS_GENDF = None
USER3_XS_GENDF = None

# Output
OUTPUT_PARQUET = '/share_snc/snc/JuanMonleon/chi2/chi2_hybrid_data_new_test_50.parquet'

In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.polynomial.legendre import legval

# Add kika root to path
_kika_path = Path().absolute().parent
if str(_kika_path) not in sys.path:
    sys.path.insert(0, str(_kika_path))

from kika.endf import read_endf
from kika.exfor import read_all_exfor
import kika.exfor as exfor
from kika.cov.mf34_covmat import MF34CovMat
from kika.cov.cross_section_covariance import CrossSectionCovariance

from scripts.exfor_utils import (
    build_exfor_cache_from_objects,
    filter_exfor_with_energy_bin,
)
from scripts.resample_AD import sample_legendre_coefficients

# Configure EXFOR database
exfor.configure(db_path=EXFOR_DB_PATH)

In [3]:
# ── Load ENDF evaluations ──────────────────────────────────────────────────────

def load_endf_mf4(filepath, label):
    """Load ENDF file and extract MF4 Legendre grid."""
    print(f"Loading {label} from {filepath}")
    endf = read_endf(filepath)
    mt = endf.get_file(4).sections[MT_NUMBER]
    energies_ev  = np.array(mt.legendre_energies)
    energies_mev = energies_ev / 1e6
    coefficients = mt.legendre_coefficients  # list of lists: [a1..aNL] per energy
    print(f"  {len(energies_mev)} grid points, E = [{energies_mev[0]:.4f}, {energies_mev[-1]:.2f}] MeV")
    return dict(
        label=label,
        energies_mev=energies_mev,
        coefficients=coefficients,  # list of arrays, one per grid point
    )

def load_endf_mf34(filepath, label):
    """Load MF34 covariance data, return None if not available."""
    try:
        covmat = MF34CovMat.from_endf(filepath, energy_unit='MeV')
        covmat = covmat.filter_by_isotope_reaction(26056, MT_NUMBER)
        print(f"  MF34: {covmat.num_matrices} covariance blocks loaded")
        return covmat
    except (ValueError, Exception) as e:
        print(f"  MF34: not available ({e})")
        return None

def load_xs_gendf(path, label):
    """Load multigroup MT=2 self-covariance from a GENDF file.

    Returns a dict {grid_ev, sigma, is_relative} where ``sigma`` is the
    per-group standard deviation (relative if is_relative=True, absolute
    in barns otherwise) and ``grid_ev`` are the multigroup boundaries.
    Returns None when the path is None or the file fails to load.

    Multigroup data is an acceptable approximation that avoids MF33 NC LTY=0
    reconstruction (sum-rule resolution for redundant MTs), which pointwise
    ENDF parsing would require. NJOY-generated GENDF files already have the
    sum rule resolved at processing time.
    """
    import numpy as np
    if path is None:
        return None
    try:
        cov = CrossSectionCovariance.from_gendf(path, energy_unit='eV')
        for i in range(cov.num_matrices):
            if (cov.isotope_rows[i] == 26056 and cov.isotope_cols[i] == 26056
                and cov.reaction_rows[i] == 2 and cov.reaction_cols[i] == 2):
                mat = cov.matrices[i]
                grid_ev = list(cov.energy_grid) if cov.energy_grid else None
                sigma = np.sqrt(np.maximum(np.diag(mat), 0.0))
                print(f"  XS GENDF ({label}): {len(sigma)} groups, "
                      f"σ_xs range=[{sigma.min():.3g}, {sigma.max():.3g}], "
                      f"is_relative={cov.is_relative[i]}")
                return dict(grid_ev=grid_ev, sigma=sigma,
                            is_relative=bool(cov.is_relative[i]))
        print(f"  XS GENDF ({label}): no MT=2 self-block found")
        return None
    except Exception as e:
        print(f"  XS GENDF ({label}): load failed — {e}")
        return None

libraries = {
    'JEFF':      {**load_endf_mf4(JEFF_FILE,  'JEFF-4.0'),
                  'mf34': load_endf_mf34(JEFF_FILE, 'JEFF-4.0'),
                  'xs':   load_xs_gendf(JEFF_XS_GENDF, 'JEFF-4.0')},
    'JENDL':     {**load_endf_mf4(JENDL_FILE, 'JENDL-5'),
                  'mf34': load_endf_mf34(JENDL_FILE, 'JENDL-5'),
                  'xs':   load_xs_gendf(JENDL_XS_GENDF, 'JENDL-5')},
    'This_work': {**load_endf_mf4(USER_FILE,  'This work'),
                  'mf34': load_endf_mf34(USER_FILE, 'This work'),
                  'xs':   load_xs_gendf(USER_XS_GENDF, 'This work')},
}

# Add optional second user file if configured
if USER2_FILE is not None:
    libraries['This_work_2'] = {**load_endf_mf4(USER2_FILE, 'This work 2'),
                                 'mf34': load_endf_mf34(USER2_FILE, 'This work 2'),
                                 'xs':   load_xs_gendf(USER2_XS_GENDF, 'This work 2')}
    print(f"\nIncluding second user evaluation: {USER2_FILE}")

# Add optional third user file if configured
if USER3_FILE is not None:
    libraries['This_work_3'] = {**load_endf_mf4(USER3_FILE, 'This work 3'),
                                 'mf34': load_endf_mf34(USER3_FILE, 'This work 3'),
                                 'xs':   load_xs_gendf(USER3_XS_GENDF, 'This work 3')}
    print(f"\nIncluding third user evaluation: {USER3_FILE}")

Loading JEFF-4.0 from /mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/26-Fe-56g.txt


/home/MONLEON-JUAN/kika/kika/endf/parsers/parse_endf.py:96: UserWarning: Skipping MF sections without parsers: [6, 8, 10, 12, 14]. Only parsing: [1, 2, 3, 4, 33, 34]
  warnings.warn(f"Skipping MF sections without parsers: {skipped_mfs}. Only parsing: {parseable_mfs}")


  3960 grid points, E = [0.0000, 45.00] MeV
  MF34: 21 covariance blocks loaded
  XS GENDF (JEFF-4.0): 56 groups, σ_xs_rel range=[0.00979, 0.0508], is_relative=True
Loading JENDL-5 from /mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/260560.jendl5


/home/MONLEON-JUAN/kika/kika/endf/parsers/parse_endf.py:96: UserWarning: Skipping MF sections without parsers: [6, 8]. Only parsing: [1, 2, 3, 4, 33, 34]
  warnings.warn(f"Skipping MF sections without parsers: {skipped_mfs}. Only parsing: {parseable_mfs}")


  1796 grid points, E = [0.0000, 20.00] MeV
  MF34: 1 covariance blocks loaded
  XS GENDF (JENDL-5): 56 groups, σ_xs_rel range=[0.0423, 0.101], is_relative=True
Loading This work from /share_snc/snc/JuanMonleon/ENDF_samples/new_test_50/26-Fe-56g_nominal_mg.endf
  4005 grid points, E = [0.0000, 45.00] MeV
  MF34: 21 covariance blocks loaded
  XS GENDF (This work): 56 groups, σ_xs_rel range=[0.00979, 0.0508], is_relative=True


In [4]:
# ── MF34 uncertainty propagation helper ───────────────────────────────────────

def propagate_mf34_uncertainty(mf34_covmat, e_mev, mu_array, endf_coeffs, c0):
    """
    Propagate MF34 Legendre covariances to dσ/dΩ uncertainties via sandwich formula.

    Returns sigma_eval array (same length as mu_array), or zeros if MF34 is None.
    """
    if mf34_covmat is None:
        return np.zeros(len(mu_array))

    L_max = len(endf_coeffs)  # endf_coeffs = [a1, a2, ..., a_L]

    # 1. Assemble L×L absolute covariance matrix at this energy
    Cov_abs = np.zeros((L_max, L_max))

    # MF34 energy grids are stored in eV (from ENDF), convert query to eV
    e_ev = e_mev * 1e6

    for idx in range(len(mf34_covmat.matrices)):
        l_r = mf34_covmat.l_rows[idx]
        l_c = mf34_covmat.l_cols[idx]
        if l_r < 1 or l_r > L_max or l_c < 1 or l_c > L_max:
            continue

        grid = np.array(mf34_covmat.energy_grids[idx])
        mat = mf34_covmat.matrices[idx]

        # Find energy bin (piecewise constant) — grid is in eV
        k = np.searchsorted(grid, e_ev, side='right') - 1
        k = np.clip(k, 0, mat.shape[0] - 1)

        cov_val = mat[k, k]  # same-energy covariance (diagonal in energy)

        # Convert relative to absolute if needed
        if mf34_covmat.is_relative[idx]:
            a_l = endf_coeffs[l_r - 1]
            a_l_prime = endf_coeffs[l_c - 1]
            cov_val *= a_l * a_l_prime

        Cov_abs[l_r - 1, l_c - 1] = cov_val
        if l_r != l_c:
            Cov_abs[l_c - 1, l_r - 1] = cov_val  # symmetric

    # 2. Build sensitivity matrix S: shape (L_max, N_mu)
    #    For l=1..L_max: S[l-1, :] = c0 * (2*l+1) * P_l(mu)
    S = np.zeros((L_max, len(mu_array)))
    for l in range(1, L_max + 1):
        coeffs_pl = np.zeros(l + 1)
        coeffs_pl[l] = 1.0  # P_l
        S[l - 1, :] = c0 * (2 * l + 1) * legval(mu_array, coeffs_pl)

    # 3. Sandwich formula: σ²[i] = S[:,i]^T @ Cov_abs @ S[:,i]
    V = Cov_abs @ S              # (L, N_mu)
    var_eval = np.sum(S * V, axis=0)  # (N_mu,)
    var_eval = np.maximum(var_eval, 0.0)  # numerical safety

    return np.sqrt(var_eval)


def propagate_mf34_uncertainty_l1(mf34_covmat, e_mev, mu_array, endf_coeffs, c0):
    """
    Like propagate_mf34_uncertainty but restricted to l=1 only.

    Only uses the (l_r=1, l_c=1) MF34 block, so sigma comes solely from
    var(a1): sigma = |c0 * 3 * P_1(mu)| * sqrt(var(a1)).
    """
    if mf34_covmat is None or len(endf_coeffs) < 1:
        return np.zeros(len(mu_array))

    # Extract var(a1) from the (1,1) MF34 block
    var_a1 = 0.0

    # MF34 energy grids are stored in eV (from ENDF), convert query to eV
    e_ev = e_mev * 1e6

    for idx in range(len(mf34_covmat.matrices)):
        l_r = mf34_covmat.l_rows[idx]
        l_c = mf34_covmat.l_cols[idx]
        if l_r != 1 or l_c != 1:
            continue

        grid = np.array(mf34_covmat.energy_grids[idx])
        mat = mf34_covmat.matrices[idx]

        # Find energy bin — grid is in eV
        k = np.searchsorted(grid, e_ev, side='right') - 1
        k = np.clip(k, 0, mat.shape[0] - 1)

        cov_val = mat[k, k]

        if mf34_covmat.is_relative[idx]:
            a1 = endf_coeffs[0]
            cov_val *= a1 * a1  # relative -> absolute

        var_a1 = cov_val
        break

    if var_a1 <= 0.0:
        return np.zeros(len(mu_array))

    # S_1 = c0 * 3 * P_1(mu) = c0 * 3 * mu
    sensitivity = c0 * 3.0 * mu_array
    var_eval = sensitivity**2 * var_a1
    return np.sqrt(np.maximum(var_eval, 0.0))

def propagate_xs_uncertainty(xs_data, e_mev, y_eval):
    """Propagate the integral cross-section relative uncertainty σ_xs(E) (from
    GENDF MF33) to the differential cross section dσ/dΩ at each μ.

    Since dσ/dΩ = c_0 · f(μ) and c_0 carries the integral XS uncertainty,
    the propagated stddev at each angle is σ_c_0_relative × |dσ/dΩ|.

    Parameters
    ----------
    xs_data : dict or None
        Output of load_xs_gendf, or None.
    e_mev : float
        Energy of the evaluation point (MeV).
    y_eval : np.ndarray
        Differential cross section dσ/dΩ at each μ.

    Returns
    -------
    np.ndarray
        Per-point σ_xs (absolute, same units as y_eval); zeros if xs_data is None.
    """
    import numpy as np
    if xs_data is None or xs_data.get('grid_ev') is None:
        return np.zeros_like(y_eval)
    grid_ev = np.asarray(xs_data['grid_ev'])
    sigma = np.asarray(xs_data['sigma'])
    e_ev = e_mev * 1e6
    # find group: piecewise constant. grid_ev has G+1 boundaries; group i covers [grid[i], grid[i+1]).
    k = np.searchsorted(grid_ev, e_ev, side='right') - 1
    k = int(np.clip(k, 0, len(sigma) - 1))
    sigma_rel = float(sigma[k])
    if not xs_data.get('is_relative', True):
        # Absolute covariance: sqrt(diag) is in absolute XS units. Convert to
        # relative by dividing by the cross section value, but since that's
        # not propagated here we use it directly as if relative — this branch
        # is rare for GENDF (which is typically relative).
        return np.abs(sigma_rel) * np.ones_like(y_eval)
    return sigma_rel * np.abs(y_eval)


In [5]:
# ── Load EXFOR data ───────────────────────────────────────────────────────────

print("Loading EXFOR data from database...")
exfor_dict = read_all_exfor(
    target=TARGET_ZAIDS, mt=MT_NUMBER, source="database",
    group_by_energy=False,
    supplementary_json_files=SUPPLEMENTARY_JSON_FILES,
    exclude_experiments=EXCLUDE_EXPERIMENTS,
)
exfor_objects = list(exfor_dict.values())
exfor_cache, sorted_exfor_energies = build_exfor_cache_from_objects(
    exfor_objects, exclude_experiments=EXCLUDE_EXPERIMENTS,
)
print(f"EXFOR cache: {len(exfor_cache)} energies, "
      f"range [{sorted_exfor_energies[0]:.4f}, {sorted_exfor_energies[-1]:.2f}] MeV")

Loading EXFOR data from database...
EXFOR cache: 22687 energies, range [0.0350, 96.00] MeV


In [6]:
# ── Main precomputation loop ──────────────────────────────────────────────────

# Use "This work" ENDF grid as reference, filtered to EXFOR energy range
ref = libraries['This_work']
exfor_emin = sorted_exfor_energies[0]
exfor_emax = sorted_exfor_energies[-1]
ref_energies = ref['energies_mev']
grid_mask = (ref_energies >= exfor_emin) & (ref_energies <= exfor_emax)
grid_energies = ref_energies[grid_mask]
grid_indices  = np.where(grid_mask)[0]
print(f"Reference grid: {len(grid_energies)} points in EXFOR range "
      f"[{exfor_emin:.4f}, {exfor_emax:.2f}] MeV")

rows = []
skipped_few_points = 0
skipped_fit_fail = 0

for gi, ref_idx in enumerate(grid_indices):
    e_mev = ref_energies[ref_idx]

    # Bin boundaries = midpoints between adjacent grid energies
    if ref_idx > 0:
        bin_lower = (ref_energies[ref_idx - 1] + e_mev) / 2
    else:
        bin_lower = 0.0
    if ref_idx < len(ref_energies) - 1:
        bin_upper = (e_mev + ref_energies[ref_idx + 1]) / 2
    else:
        bin_upper = float('inf')

    # Get ALL EXFOR data in this bin (for evaluation points)
    df_all, exp_info_all, kw_all, _, _ = filter_exfor_with_energy_bin(
        exfor_cache=exfor_cache,
        sorted_energies=sorted_exfor_energies,
        bin_lower_mev=bin_lower,
        bin_upper_mev=bin_upper,
        target_energy_mev=e_mev,
        m_proj_u=M_PROJ_U, m_targ_u=M_TARG_U,
        dedupe_per_experiment=True,
        exclude_experiments=EXCLUDE_EXPERIMENTS,
        min_relative_uncertainty=MIN_STAT_RELATIVE_UNCERTAINTY,
        normalize_by_n_points=True,
        max_experiment_weight_fraction=0.5,
    )

    if df_all.empty or len(df_all) < 3:
        skipped_few_points += 1
        continue

    # Add experiment_id column if missing
    if 'experiment_id' not in df_all.columns:
        df_all['experiment_id'] = df_all['entry'] + df_all['subentry']

    # Derive is_natural from reaction column
    if 'reaction' in df_all.columns:
        df_all['is_natural'] = df_all['reaction'].str.contains('FE-0', case=False, na=False)
    else:
        df_all['is_natural'] = False

    # Determine which subentry to use for JEFF/JENDL c0 fit
    post_filter_sub = KINNEY_SUBENTRY if e_mev < ENERGY_THRESHOLD_MEV else SMITH_SUBENTRY

    # Process each library
    for lib_key, lib_data in libraries.items():
        lib_energies = lib_data['energies_mev']
        lib_coeffs   = lib_data['coefficients']

        # Find closest grid point in this library
        lib_idx = int(np.argmin(np.abs(lib_energies - e_mev)))
        endf_coeffs = np.array(lib_coeffs[lib_idx])  # [a1, a2, ..., a_L]
        L_max = len(endf_coeffs)

        # Prepare fitting subset
        if lib_key in ('JEFF', 'JENDL'):
            # Post-filter to Kinney or Smith subentry
            entry_str = post_filter_sub[:5]
            sub_str   = post_filter_sub[5:]
            mask = (df_all['entry'] == entry_str) & (df_all['subentry'] == sub_str)
            if mask.any():
                fit_df = df_all[mask].reset_index(drop=True)
                fit_kw = kw_all[mask.values] if kw_all is not None and len(kw_all) > 0 else None
            else:
                fit_df = df_all
                fit_kw = kw_all
        else:
            # This work: use all experiments
            fit_df = df_all
            fit_kw = kw_all

        if len(fit_df) < 2:
            skipped_fit_fail += 1
            continue

        # Fit c0 from EXFOR
        try:
            coef_df, fit_info = sample_legendre_coefficients(
                df=fit_df, value_col="value", unc_col="unc", mu_col="mu",
                degree=L_max, n_samples=1,
                external_weights=fit_kw,
                ridge_lambda=1e-6, rescale_unc_by_chi2=True,
            )
        except Exception:
            skipped_fit_fail += 1
            continue

        c0 = coef_df['c0'].values[0]
        if c0 <= 0 or not np.isfinite(c0):
            skipped_fit_fail += 1
            continue

        # Build hybrid coefficients: [c0, c0*3*a1, c0*5*a2, ...]
        hybrid_coeffs = [c0]
        for l in range(1, L_max + 1):
            a_l = endf_coeffs[l - 1]
            hybrid_coeffs.append(c0 * (2 * l + 1) * a_l)

        # Evaluate at ALL EXFOR points in bin
        mu_data = df_all['mu'].values
        y_eval = legval(mu_data, hybrid_coeffs)

        # Propagate MF34 covariance to dσ/dΩ uncertainty at each μ
        mf34 = lib_data.get('mf34')
        sigma_eval_mf34 = propagate_mf34_uncertainty(mf34, e_mev, mu_data, endf_coeffs, c0)
        sigma_eval_l1   = propagate_mf34_uncertainty_l1(mf34, e_mev, mu_data, endf_coeffs, c0)

        # Propagate integral XS uncertainty (GENDF / MF33) to dσ/dΩ
        xs_data = lib_data.get('xs')
        sigma_eval_xs = propagate_xs_uncertainty(xs_data, e_mev, y_eval)

        # Combined evaluation σ (MF34 ⊕ XS in quadrature)
        sigma_eval = np.sqrt(sigma_eval_mf34**2 + sigma_eval_xs**2)

        # Collect rows. Per-point experimental uncertainty is now decomposed
        # into σ_stat (from manifest, in df_all['unc']) and σ_sys (per-row
        # absolute b/sr from manifest, in df_all['error_sys']) — combined as
        # σ_exp = √(σ_stat² + σ_sys²) to match the chi² convention.
        sigma_stat_arr = df_all['unc'].to_numpy(dtype=float)
        if 'error_sys' in df_all.columns:
            sigma_sys_arr = df_all['error_sys'].to_numpy(dtype=float)
        else:
            # Legacy fallback: derive from sigma_sys_relative if present
            rel = df_all['sigma_sys_relative'].to_numpy(dtype=float) if 'sigma_sys_relative' in df_all.columns else np.zeros(len(df_all))
            sigma_sys_arr = rel * np.abs(df_all['value'].to_numpy(dtype=float))
        sigma_exp_total_arr = np.sqrt(sigma_stat_arr**2 + sigma_sys_arr**2)

        for i in range(len(df_all)):
            row = df_all.iloc[i]
            rows.append({
                'energy_mev':         e_mev,
                'mu':                 row['mu'],
                'y_exp':              row['value'],
                'sigma_exp_stat':     float(sigma_stat_arr[i]),
                'sigma_exp_sys':      float(sigma_sys_arr[i]),
                'sigma_exp':          float(sigma_exp_total_arr[i]),  # √(stat² + sys²)
                'sigma_eval_mf34':    float(sigma_eval_mf34[i]),
                'sigma_eval_xs':      float(sigma_eval_xs[i]),
                'sigma_eval':         float(sigma_eval[i]),  # √(mf34² + xs²)
                'sigma_eval_l1':      float(sigma_eval_l1[i]),
                'experiment_id':      row['experiment_id'],
                'author':             row['author'],
                'year':               row['year'],
                'is_natural':         row['is_natural'],
                'library':            lib_key,
                'y_eval':             y_eval[i],
                'c0':                 c0,
                'L_max':              L_max,
            })

print(f"\nDone. Collected {len(rows)} rows.")
print(f"Skipped {skipped_few_points} grid points with < 3 EXFOR points.")
print(f"Skipped {skipped_fit_fail} library/grid combos due to fit failures.")

Reference grid: 3988 points in EXFOR range [0.0350, 96.00] MeV

Done. Collected 149768 rows.
Skipped 372 grid points with < 3 EXFOR points.
Skipped 350 library/grid combos due to fit failures.


In [7]:
# ── Save to parquet ───────────────────────────────────────────────────────────

df = pd.DataFrame(rows)
df.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Saved to: {OUTPUT_PARQUET}")
print(f"Shape: {df.shape}")
print(f"Libraries: {df['library'].unique().tolist()}")
print(f"Energy range: [{df['energy_mev'].min():.4f}, {df['energy_mev'].max():.2f}] MeV")
print(f"Experiments: {df['experiment_id'].nunique()}")
print(f"Grid points: {df['energy_mev'].nunique()}")
print(f"\nRows per library:")
print(df.groupby('library').size().to_string())

Saved to: /share_snc/snc/JuanMonleon/chi2/chi2_hybrid_data_new_test_50.parquet
Shape: (149768, 18)
Libraries: ['JEFF', 'JENDL', 'This_work']
Energy range: [0.0350, 14.50] MeV
Experiments: 109
Grid points: 3508

Rows per library:
library
JEFF         50412
JENDL        48944
This_work    50412
